# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

# Print dataset metadata summary
print(f"{metadata['name']}: {metadata['description']}")
print(f"Date Published: {metadata.get('datePublished', 'unknown')}")
print(f"License: {metadata.get('license', 'unknown')}")
print(f"Identifier: {metadata.get('identifier', 'unknown')}")
print(f"Available keywords: {metadata.get('keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant schemas index record sets, fields, and columns by their `@id`. We'll enumerate all available record sets and list their fields and columns.

In [ ]:
# List record sets
record_sets = dataset.record_sets
print('Available Record Sets:')
for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']}")
    print(f"    Name: {rs.get('name', 'N/A')}")
    if 'fields' in rs:
        print("    Fields:")
        for f in rs['fields']:
            print(f"      Field @id: {f['@id']} -- Name: {f.get('name', 'N/A')}")
    if 'columns' in rs:
        print("    Columns:")
        for c in rs['columns']:
            print(f"      Column @id: {c['@id']} -- Name: {c.get('name', 'N/A')}")
    print()

# Preview a few records from each record set
for rs in record_sets:
    print(f"Preview for RecordSet @id: {rs['@id']} (Name: {rs.get('name', 'N/A')}):")
    for i, rec in enumerate(dataset.records(record_set=rs['@id'])):
        print(rec)
        if i >= 2:
            break
    print("---\n")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

**All entities are referenced by their `@id`.**

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load records for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df

# Show all columns for the first record set
primary_record_set_id = record_set_ids[0] if record_set_ids else None
if primary_record_set_id:
    print(f"Columns for RecordSet @id {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

**All operations reference fields/columns by their `@id`.**

In [ ]:
# Pick a numeric field (example: Age) using @id
# Find numeric field candidates
def find_numeric_candidate(rs):
    # scan for 'fields' or 'columns' that are numeric
    numeric_types = ['schema:Integer', 'schema:Number', 'schema:Float']
    for f in rs.get('fields', []):
        if f.get('dataType', None) in numeric_types or f.get('@type', None) in numeric_types:
            return f['@id']
    for c in rs.get('columns', []):
        if c.get('dataType', None) in numeric_types or c.get('@type', None) in numeric_types:
            return c['@id']
    return None

numeric_field_id = None
group_field_id = None
for rs in record_sets:
    candidate = find_numeric_candidate(rs)
    if candidate:
        numeric_field_id = candidate
        if rs.get('fields'):
            # Try grouping by a categorical field
            for f in rs['fields']:
                if f.get('dataType') == 'schema:Text':
                    group_field_id = f['@id']
                    break
        break

# If not found, fall back to first column
if not numeric_field_id and primary_record_set_id:
    # try to guess based on column names for demo purposes
    for col in dataframes[primary_record_set_id].columns:
        if 'age' in col.lower():
            numeric_field_id = col
            break
if not group_field_id and primary_record_set_id:
    # try to guess based on column names
    for col in dataframes[primary_record_set_id].columns:
        if 'sex' in col.lower() or 'gender' in col.lower():
            group_field_id = col
            break

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

if numeric_field_id and primary_record_set_id:
    df = dataframes[primary_record_set_id]
    if numeric_field_id in df.columns:
        threshold = 50
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Example: Histogram of the selected numeric field and boxplot grouped by the selected categorical field.

In [ ]:
if numeric_field_id and primary_record_set_id:
    df = dataframes[primary_record_set_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(6,4))
        sns.histplot(df[numeric_field_id], bins=15, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()

        if group_field_id and group_field_id in df.columns:
            plt.figure(figsize=(8,4))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides detailed clinicopathological and molecular characteristics of cancer survivors with second primary colorectal cancer.
- Data fields are referenced by their `@id`, ensuring precise mapping between schema and records.
- By exploring numeric fields (e.g., age) and grouping by categorical values (e.g., sex), we observe distributions that may inform clinical and biomarker research.
- `mlcroissant` streamlines loading and processing of FAIR datasets, enabling transparent and reproducible analysis.

*End of notebook.*